# Benchmarking Approximate Nearest Neighbour SearchCS4.406 — Assignment 3. This notebook runs **every experiment** in the report.It is a front end over `src/annbench/`, not a second implementation: each cellcalls the same functions `run_all.sh` calls, so a number produced here and anumber produced by the script are the same number, written to the same`results/results.csv`.**How to use it**1. Run the setup cell below. Set `QUICK = True` for a laptop-sized run   (100K-vector subsets, coarse grid, a few minutes) or `QUICK = False` for the   full run behind the report (hours).2. Run the cells in order. Each experiment appends to `results/results.csv` and   **skips rows already there**, so you can stop and resume freely, or re-run a   cell without redoing its work. Pass `--force` to recompute.3. The last section regenerates every figure and table from the CSV alone.**Before you start:** `bash scripts/download_data.sh` fetches the two datasetsinto `data/`. Part 4 needs no download — it generates its own data.

## SetupThread limits must be set **before** NumPy or FAISS are imported — for some BLAS builds, setting them afterwards is too late and the single-thread timing protocol would silently not apply.

In [ ]:
import os

# ---- must come before any numpy / faiss import -------------------------
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
# ------------------------------------------------------------------------

import sys
from pathlib import Path

# Run from anywhere: find the repo root (the directory holding src/annbench).
ROOT = Path.cwd()
while not (ROOT / "src" / "annbench").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

QUICK = True          # <-- False for the full run behind the report
FORCE = False         # <-- True to recompute rows already in results.csv

CONFIG  = "configs/quick.yaml" if QUICK else "configs/full.yaml"
RESULTS = "results/results.csv"
FIGURES = "results/figures"
TABLES  = "results/tables"
EXTRA   = ["--force"] if FORCE else []
for d in (FIGURES, TABLES, "data/cache"):
    os.makedirs(d, exist_ok=True)

from annbench.config import load_config
from annbench.env import collect_env

cfg = load_config(CONFIG)
print(f"repo    : {ROOT}")
print(f"config  : {CONFIG}  ({'QUICK' if QUICK else 'FULL'} run)")
for k, v in sorted(collect_env(str(ROOT)).items()):
    print(f"  {k:18s} {v}")

## Part 0 — unit testsThe recall function is hand-written, so it gets tested before it is trusted.The interesting cases are the ties at the 10th neighbour, where the naiveset-intersection formula gives the wrong answer.

In [ ]:
!python3 -m pytest tests -q

## Part 1 — exact baselineThree things: check our exact search against the neighbour ids shipped witheach dataset, time exact search one-query-at-a-time and batched, and write thebaseline row that every later speed-up is measured against.Expect the naive agreement with the shipped ground truth to be *slightly* below1.0 and the tie-aware recall to be exactly 1.0 — that gap is the evidence thatthe disagreements are ties rather than errors.

In [ ]:
from annbench.experiments import part1_exact

part1_exact.main(["--config", CONFIG, "--results", RESULTS, "--tables", TABLES] + EXTRA)

In [ ]:
from IPython.display import Markdown, display

for name in ("part1_verification", "part1_timing"):
    display(Markdown(open(f"{TABLES}/{name}.md").read()))

### Why the two timings differ so much- **One query at a time** is a matrix–vector product (BLAS level 2). Every  query streams the whole database through the CPU once, so throughput is  bounded by memory bandwidth and each database vector is touched for a single  multiply-add pass before being evicted.- **All queries in one batch** is a matrix–matrix product (BLAS level 3). A  tile of the database is loaded into cache and reused against every query in  the block, so the same memory traffic serves thousands of queries. Per-call  overhead is amortised too.Both are measured on one thread, so the gap is this effect and not parallelism.

## Part 2 — HNSW, IVF-Flat and IVF-PQ on both datasetsBuilds every configured (index, build-parameter) combination and sweeps itssearch parameter (`efSearch` or `nprobe`) until recall saturates.This is the long cell. It is resumable: stop it whenever you like and re-run —it picks up from what is already in `results.csv`.

In [ ]:
from annbench.experiments import part2_benchmark

part2_benchmark.main(["--config", CONFIG, "--results", RESULTS] + EXTRA)

### The trade-off curvesThe frontier is selected on the **tuning** queries and drawn at **held-out**recall. Faint markers are every measured configuration; the line is the Paretoenvelope — the best this index type can do at each accuracy.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image, display

from annbench.plots import plot_tradeoff
from annbench.results import load_results

df = load_results(RESULTS)
p2 = df[df["experiment"] == "part2"]

for dataset in sorted(p2["dataset"].unique()):
    exact = df[(df["experiment"] == "part1") & (df["dataset"] == dataset)]
    exact_qps = float(exact["qps_median"].iloc[0]) if len(exact) else None
    path = plot_tradeoff(p2, dataset, f"{FIGURES}/part2_tradeoff_{dataset}.png",
                         exact_qps=exact_qps,
                         title=f"{dataset}: recall@10 vs throughput")
    display(Image(path))

In [ ]:
from IPython.display import Markdown, display

from annbench.experiments import make_tables

make_tables.main(["--results", RESULTS, "--tables", TABLES])
for name in ("part2_qps_at_recall", "part2_cost_at_recall"):
    display(Markdown(open(f"{TABLES}/{name}.md").read()))

An index showing **"not reached"** at recall 0.95 has not failed to bemeasured — it has hit a ceiling. For IVF-PQ that ceiling is set by quantisationerror, not by `nprobe`: even scanning every cell, the candidates are rankedusing distances computed from 8-bit codes, so the ordering is wrong no matterhow much of the index is examined. More probing cannot repair a corrupteddistance function; only re-ranking with the original vectors can, which is whatthe `IVF-PQ refine` configuration does — and it needs the full vectors inmemory, which is the memory saving handed straight back.

## Part 3 — scaling on SIFTHow does the cost of reaching recall 0.95 grow with the database?Two things this cell does that are easy to get wrong:- **Ground truth is recomputed for every subset.** The ids in the file index  into the full 1M database; at 100K most of them are not even in the index, so  reusing them would measure the subsetting rather than the index.- **`nlist` is retuned at every size** (≈ 4·√N). Holding it fixed would measure  "an index tuned for 1M, run on 100K".The hypothesis for this experiment is in `hypothesis.md`, committed before theexperiment was run.

In [ ]:
from annbench.experiments import part3_scaling

part3_scaling.main(["--config", CONFIG, "--results", RESULTS] + EXTRA)

In [ ]:
from IPython.display import Image, Markdown, display

from annbench.experiments import make_tables
from annbench.plots import plot_scaling
from annbench.results import load_results

df = load_results(RESULTS)
display(Image(plot_scaling(df, f"{FIGURES}/part3_scaling.png", target=0.95,
                           title="Part 3: cost of recall 0.95 as the database grows")))

make_tables.main(["--results", RESULTS, "--tables", TABLES])
display(Markdown(open(f"{TABLES}/part3_scaling.md").read()))

## Part 4 — intrinsic dimensionality, on generated data**The question:** how does the number of directions the data actually varies inchange what each index can do, at a *fixed* ambient dimension?Vectors are generated on a random k-dimensional linear subspace of a fixed128-dimensional space, plus a small isotropic noise floor, fork ∈ {2, 8, 32, 128}. Every condition has identical vector size, distance cost,index memory and PQ layout — only k changes. **k = 128 = d is the control**(isotropic Gaussian data).Varying the ambient dimension instead would confound geometry with bytes pervector and arithmetic cost; this design does not.The generator is seeded, so the same seed gives byte-identical data on anymachine. Hypothesis in `hypothesis.md`, committed first.

In [ ]:
from annbench.experiments import part4_intrinsic

part4_intrinsic.main(["--config", CONFIG, "--results", RESULTS, "--tables", TABLES] + EXTRA)

In [ ]:
from IPython.display import Image, Markdown, display

from annbench.plots import plot_part4
from annbench.results import load_results

df = load_results(RESULTS)
p4 = df[df["experiment"] == "part4"]
dim = int(p4["dim"].dropna().iloc[0])
n   = int(p4["n_vectors"].dropna().iloc[0])

display(Image(plot_part4(df, f"{FIGURES}/part4_intrinsic_dimension.png", target=0.95,
                         title=f"Part 4: intrinsic dimensionality at fixed ambient d={dim}, n={n:,}")))
display(Markdown(open(f"{TABLES}/part4_contrast.md").read()))

Read the right-hand panel with both lines at once: the solid line is thethroughput needed to reach recall 0.95, and the dashed line is the best recallthe sweep reached at all. **Where the dashed line falls below 0.95 the solidline is necessarily absent** — the index cannot reach the target at any speed.That absence is the finding, not a gap in the data.Relative contrast is the sanity check on the mechanism: it is the mean distancefrom a query to a random database vector divided by its distance to the truenearest neighbour. Values approaching 1 mean nothing is distinctly closest,which is the regime where partition-based pruning stops being able to help.

## Is timing on the tuning split safe?Throughput is measured on the tuning queries while recall is reported on theheld-out queries, on the argument that QPS is a property of the index and itssearch parameter rather than of which queries are used. That is an argument,not a fact, so this measures it: the same configurations timed on both splits,with the difference shown next to the run-to-run spread of the repeats.If the split difference sits inside the repeat spread, the choice costsnothing. This is one of the two "ways the benchmark could mislead" that Part 5asks about — tested rather than merely acknowledged.

In [ ]:
!python3 scripts/check_timing_split.py --config {CONFIG} --dataset sift

## Regenerate everything from `results.csv`The measurements write only to `results.csv`; the figures and tables read onlyfrom it. These two commands rebuild every figure and table in the report fromthat file alone — which is what makes the reproduction claim structural ratherthan aspirational.

In [ ]:
from annbench.experiments import make_figures, make_tables
from annbench.results import load_results

make_figures.main(["--results", RESULTS, "--figures", FIGURES])
make_tables.main(["--results", RESULTS, "--tables", TABLES])

df = load_results(RESULTS)
print(f"\n{len(df)} rows in {RESULTS}")
print(df.groupby(["experiment", "dataset", "index_type"]).size().to_string())

In [ ]:
# A look at the raw results table: everything behind every number.
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)
df[["experiment", "dataset", "variant", "n_vectors", "index_type", "params_json",
    "search_param", "recall_tune", "recall_test", "qps_median", "n_queries_timed",
    "build_time_s", "bytes_per_vector"]].head(25)